In [ ]:
#CELDA 1
#from google.colab import drive
#drive.mount('/content/drive')

TASK_CSV = "D:/Documentos/ExpLLMs/corpus_ext_task.csv"
PLUS_CSV = "D:/Documentos/ExpLLMs/corpus_ext_plus.csv"
RUNS_DIR = "D:/Documentos/ExpLLMs/runs"
url = "http://localhost:1234"
# --- MODEL SELECTOR ---
MODELS = {
    0: dict(
        model_id="qwen-qwen2.5-14b-instruct-1m",
        base_url=url
    ),
    1: dict(
        model_id="openai/gpt-oss-20b",
        base_url=url
    ),
    2: dict(
        model_id="mistral-small-3.2-24b-instruct-2506",
        base_url=url
    )
}
ACTIVE_MODEL = 0

MODEL_ID = MODELS[ACTIVE_MODEL]["model_id"]
BASE_URL = MODELS[ACTIVE_MODEL]["base_url"]
API_KEY  = None  # LM Studio local, no requiere API key

import re
MODEL_SLUG = re.sub(r'[^A-Za-z0-9_.-]+', '_', MODEL_ID)  # p.ej. "openai/gpt-oss-20b" -> "openai_gpt-oss-20b"

SYSTEM_PROMPT = (
    "You are a helpful linguistics assistant.\n"
    "Write only one line as: FINAL: <target_sentence>.\n"
    "Do not explain. Do not translate. Do not add or remove punctuation unless required by Change or by the examples.\n"
    "If unsure, still output a single best target sentence only."
)

# --- EXPERIMENT SETUP ---
# Sin splits: tratamos todo el corpus como un solo pool
COMPLEXITIES   = [1, 2]
K_LIST         = [5,10,15]        # solo 5-shots
WITH_HINTS     = [False, True]

# --- DECODING ---
DECODE = dict(temperature=0, top_k=1, max_tokens=64)

# --- RETRIEVAL POLICY ---
# Usamos todo el corpus; garantizamos al menos 1 shot del mismo cluster del test si existe.
EXCLUDE_SAME_CLUSTER = False
PRIORITY = ["exact_change", "primary_change", "similarity"]
SUBSET_CONSTRAINT = True
MAX_CANDIDATES_PER_STAGE = 2000

# --- THROTTLE / LIMITS ---
MAX_TESTS_PER_SIZE = 80
MAX_PER_PRIMARY    = 10
FILTER_PRIMARY     = []
SAMPLE_FRACTION    = None

RANDOM_SEED = 430


In [8]:
# CELDA 2
import pandas as pd, numpy as np, json, unicodedata, re, time, random, os, requests
from collections import defaultdict, Counter
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

def nfc(s):
    return unicodedata.normalize("NFC", s or "")

def parse_change_to_dict(change_str):
    # Robust to NaN/None
    if change_str is None or (isinstance(change_str, float) and pd.isna(change_str)):
        return {}
    s = str(change_str).strip()
    if s == "{}":
        return {}
    s = s.strip("{}").strip()
    if not s:
        return {}
    out = {}
    parts = [p for p in s.split(",") if p.strip()]
    for p in parts:
        if ":" in p:
            k, v = p.split(":", 1)
            out[k.strip()] = v.strip()
    return out

def change_key_tuple(change_dict):
    order = ["PERSON","NUMBER","POSS","TENSE","ASPECT","PERSON_OBJ","MODE","TYPE","SUBTYPE","EVID"]
    keys = sorted(change_dict.keys(), key=lambda k:(order.index(k) if k in order else 999, k))
    return tuple((k, change_dict[k]) for k in keys)

def change_keys_only(change_dict):
    order = ["PERSON","NUMBER","POSS","TENSE","ASPECT","PERSON_OBJ","MODE","TYPE","SUBTYPE","EVID"]
    keys = sorted(change_dict.keys(), key=lambda k:(order.index(k) if k in order else 999, k))
    return tuple(keys)

def primary_change(change_dict):
    order = ["PERSON","NUMBER","POSS","TENSE","ASPECT","PERSON_OBJ","MODE","TYPE","SUBTYPE","EVID"]
    for k in order:
        if k in change_dict:
            return k
    return "NONE"

# Load corpora
task_df = pd.read_csv(TASK_CSV)
plus_df = pd.read_csv(PLUS_CSV)

# Ensure change_dict
for df_ in (task_df, plus_df):
    if "change" in df_.columns:
        df_["change_dict"] = df_["change"].map(parse_change_to_dict)
    if "change_obj" in df_.columns:
        df_["change_dict"] = df_["change_obj"].map(parse_change_to_dict)

# Derived columns
task_df["size"] = task_df["change_dict"].map(lambda d: len(d))
task_df["primary"] = task_df["change_dict"].map(primary_change)

# Join hints & cluster_id from plus_df using (source, target, change)
if "hints" in plus_df.columns:
    key_plus = plus_df.assign(key=lambda x: x["source_quz"].astype(str)+"|||"+x["target_quz"].astype(str)+"|||"+x["change_obj"].astype(str))[
        ["key","hints","cluster_id"]]
else:
    key_plus = plus_df.assign(hints=None).assign(key=lambda x: x["source_quz"].astype(str)+"|||"+x["target_quz"].astype(str)+"|||"+x["change_obj"].astype(str))[
        ["key","hints","cluster_id"]]

task_df = task_df.assign(key=lambda x: x["source_quz"].astype(str)+"|||"+x["target_quz"].astype(str)+"|||"+x["change"].astype(str))
task_df = task_df.merge(key_plus, on="key", how="left")

In [9]:
#CELDA 3
# Pools using the whole corpus (no splits)
task_df["ckey"]  = task_df["change_dict"].map(change_key_tuple)
task_df["ckeys"] = task_df["change_dict"].map(change_keys_only)

# Stable base order to avoid incidental variability
stable_cols = [c for c in ["cluster_id","id"] if c in task_df.columns]
pool_all = task_df.sort_values(stable_cols).reset_index(drop=True).copy()

# Fast access dictionaries
pool_exact   = defaultdict(list)
pool_primary = defaultdict(list)
for _, row in pool_all.iterrows():
    pool_exact[row["ckey"]].append(row)
    pool_primary[row["primary"]].append(row)

# TF-IDF on whole pool (deterministic by stable sort above)
if len(pool_all) > 0:
    tfidf      = TfidfVectorizer(min_df=1).fit(pool_all["source_quz"].astype(str).tolist())
    train_vecs = tfidf.transform(pool_all["source_quz"].astype(str).tolist())
    print(f"[TFIDF] Vocab size={len(tfidf.vocabulary_)} | vecs={train_vecs.shape}")
else:
    tfidf = None
    train_vecs = None
    print("[TFIDF] pool_all vacío; se omitirá similaridad.")

def is_subset_change(kshot_change, test_change):
    if not SUBSET_CONSTRAINT:
        return True
    for k, v in kshot_change.items():
        if k not in test_change: return False
        if test_change[k] != v:  return False
    return True

def _row_key(series_like):
    if "key" in series_like:
        return str(series_like["key"])
    src = str(series_like.get("source_quz", ""))
    tgt = str(series_like.get("target_quz", ""))
    chg = str(series_like.get("change", ""))
    return src + "|||" + tgt + "|||" + chg

def _safe_id(series_like):
    try:
        return int(series_like.get("id", -1))
    except Exception:
        return -1

def _cluster(series_like):
    return series_like.get("cluster_id", None)

def _det_seed(test_id, K):
    # Deterministic RNG per (seed, test_id, K)
    return (int(RANDOM_SEED) << 20) ^ (int(test_id) << 4) ^ int(K)

def _det_shuffle(seq, rng):
    seq = list(seq)
    rng.shuffle(seq)
    return seq

def _order_cluster_first(cands, test_cluster):
    if test_cluster is None:
        return list(cands)
    same, diff = [], []
    for c in cands:
        (same if _cluster(c) == test_cluster else diff).append(c)
    # Within groups, sort stably by (id,row_key) to be deterministic before RNG
    def _key(c): return (_safe_id(c), _row_key(c))
    same = sorted(same, key=_key); diff = sorted(diff, key=_key)
    return same + diff

def _exists_valid_same_cluster(test_row):
    tc = _cluster(test_row)
    if tc is None:
        return False
    tkey = _row_key(test_row)
    for _, c in pool_all.iterrows():
        if _cluster(c) != tc: continue
        if _row_key(c) == tkey: continue
        if is_subset_change(c["change_dict"], test_row["change_dict"]):
            return True
    return False

def select_kshots(test_row, K):
    """Deterministic selection per (test_id, K).
       Leave-one-out; ensure ≥1 same-cluster shot if possible; complete with other clusters."""
    test_change  = test_row["change_dict"]
    test_ckey    = test_row["ckey"]
    test_primary = test_row["primary"]
    test_cluster = _cluster(test_row)
    test_key     = _row_key(test_row)
    test_id      = _safe_id(test_row)

    rng = random.Random(_det_seed(test_id, K))

    def valid_candidate(c):
        if _row_key(c) == test_key:  # leave-one-out
            return False
        return is_subset_change(c["change_dict"], test_change)

    shots = []
    selected_keys = {test_key}

    def add_candidates(cands, needed):
        added = 0
        for c in cands:
            if added >= needed: break
            k = _row_key(c)
            if k in selected_keys: continue
            if not valid_candidate(c): continue
            shots.append(c); selected_keys.add(k); added += 1
        return added

    needed = K

    # 1) EXACT
    if "exact_change" in PRIORITY and needed > 0:
        base = pool_exact.get(test_ckey, [])[:MAX_CANDIDATES_PER_STAGE]
        cands = _order_cluster_first(base, test_cluster)
        cands = _det_shuffle(cands, rng)
        needed -= add_candidates(cands, needed)

    # 2) PRIMARY
    if "primary_change" in PRIORITY and needed > 0:
        base = pool_primary.get(test_primary, [])[:MAX_CANDIDATES_PER_STAGE]
        cands = _order_cluster_first(base, test_cluster)
        cands = _det_shuffle(cands, rng)
        needed -= add_candidates(cands, needed)

    # 3) SIMILARITY on whole pool
    if "similarity" in PRIORITY and needed > 0 and tfidf is not None and train_vecs is not None and len(pool_all) > 0:
        q    = tfidf.transform([str(test_row["source_quz"])])
        sims = cosine_similarity(q, train_vecs).ravel()
        idx  = np.argsort(-sims)[:min(200, len(pool_all))]
        base = [pool_all.iloc[i] for i in idx]
        cands = _order_cluster_first(base, test_cluster)
        # No shuffle here; TF-IDF already gives a deterministic ranking on our stable pool
        needed -= add_candidates(cands, needed)

    # Ensure ≥1 same-cluster shot if possible
    if test_cluster is not None:
        has_same = any(_cluster(s) == test_cluster for s in shots)
        if not has_same and _exists_valid_same_cluster(test_row):
            # Find one valid same-cluster; replace last if already full
            # Deterministic scan over stable pool
            for _, c in pool_all.iterrows():
                if _cluster(c) != test_cluster: continue
                if not valid_candidate(c): continue
                if len(shots) >= K and len(shots) > 0:
                    shots[-1] = c
                else:
                    shots.append(c)
                break

    return shots[:K]

[TFIDF] Vocab size=579 | vecs=(6344, 579)


In [10]:
# CELDA 4
def hints_for_test_row(row):
    # Si plus_df ya traía "hints" generados, úsalo.
    h = row.get("hints", None)
    if pd.isna(h) or h is None:
        return None
    try:
        arr = json.loads(h)
        if not isinstance(arr, list):
            return None
        # formato del bloque HINTS (exactamente como vienes usando)
        block = ["HINTS (apply strictly; morpheme notation is descriptive; output surface forms only):"]
        for line in arr:
            block.append(f"- {line}")
        return "\n".join(block)
    except Exception:
        return None

def build_user_prompt(test_row, kshots, with_hints=True):
    header = (
        "This is a linguistic puzzle with examples.\n"
        "You are given pairs of (Source, Change -> Target).\n"
        "Generate ONLY the target sentence for the final example.\n"
        "Write only one line as: FINAL: <target_sentence>\n"
        "Do not echo the table. Do not explain anything.\n"
    )
    lines = [header]

    if with_hints:
        hb = hints_for_test_row(test_row)
        if hb:
            lines.append("\n" + hb + "\n")

    lines.append("Source,Change,Target")

    # K-shot lines (con Target lleno)
    for c in kshots:
        src = nfc(c["source_quz"]); chg = nfc(c["change"]); tgt = nfc(c["target_quz"])
        lines.append(f"{src},{{{', '.join([f'{k}:{v}' for k,v in c['change_dict'].items()])}}},{tgt}")

    # Fila final: target vacío
    src = nfc(test_row["source_quz"]); chg = nfc(test_row["change"])
    lines.append(f"{src},{{{', '.join([f'{k}:{v}' for k,v in test_row['change_dict'].items()])}}},")

    return "\n".join(lines)

In [18]:
# CELDA 5
def call_lmstudio(user_prompt: str, system_prompt: str = SYSTEM_PROMPT, model_id: str = MODEL_ID):
    url = f"{BASE_URL.rstrip('/')}/v1/chat/completions"
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    payload = {
        "model": model_id,
        "temperature": DECODE["temperature"],
        "max_tokens": DECODE["max_tokens"],
        "top_k": DECODE["top_k"],
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }
    # Special case: openai/gpt-oss-20b -> no max_tokens limit
    if model_id == "openai/gpt-oss-20b":
        payload.pop("max_tokens", None)

    # Long-read timeout for heavy reasoning models
    long_timeout_models = {
        "openai/gpt-oss-20b",
        "mistral-small-3.2-24b-instruct-2506",
    }
    req_timeout = (10, 900) if model_id in long_timeout_models else 120

    t0 = time.time()
    r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=req_timeout)
    latency_ms = int((time.time() - t0) * 1000)

    r.raise_for_status()
    txt = r.json()["choices"][0]["message"]["content"]

    m = re.search(r"^\s*FINAL\s*:\s*(.*)$", txt, flags=re.IGNORECASE|re.MULTILINE)
    pred = nfc(m.group(1).strip()) if m else ""
    return pred, txt, latency_ms

In [19]:
# CELDA 6
# Tests on the whole pool (no splits)
tests = task_df.copy()

def stable_sample(df, n):
    if len(df) <= n: return df
    return df.sample(n=n, random_state=RANDOM_SEED)

import os
os.makedirs(RUNS_DIR, exist_ok=True)

run_rows = []

for size in COMPLEXITIES:
    tests_size = tests[tests["size"] == size].copy()

    if FILTER_PRIMARY:
        tests_size = tests_size[tests_size["primary"].isin(FILTER_PRIMARY)].copy()

    if tests_size.empty:
        print(f"[AVISO] No hay tests para size={size} con filtro {FILTER_PRIMARY}")
        continue

    if SAMPLE_FRACTION is not None and 0 < SAMPLE_FRACTION < 1:
        tests_size = tests_size.sample(frac=SAMPLE_FRACTION, random_state=RANDOM_SEED)

    if MAX_PER_PRIMARY and MAX_PER_PRIMARY > 0:
        chunks = []
        for prim, sub in tests_size.groupby("primary"):
            n_take = min(MAX_PER_PRIMARY, len(sub))
            chunks.append(sub.sample(n=n_take, random_state=RANDOM_SEED))
        tests_size = pd.concat(chunks, ignore_index=True)

    if MAX_TESTS_PER_SIZE and MAX_TESTS_PER_SIZE > 0 and len(tests_size) > MAX_TESTS_PER_SIZE:
        tests_size = tests_size.sample(n=MAX_TESTS_PER_SIZE, random_state=RANDOM_SEED)

    tests_iter = tests_size.reset_index(drop=True)
    print(f"[INFO] size={size} | tests seleccionados={len(tests_iter)} "
          f"(filtro primary={FILTER_PRIMARY or 'ALL'}, max/primary={MAX_PER_PRIMARY}, max/size={MAX_TESTS_PER_SIZE})")

    test_ids = [int(r.get("id", idx)) for idx, r in tests_iter.iterrows()]

    for K in K_LIST:
        run_dir = os.path.join(RUNS_DIR, MODEL_SLUG, f"K{K}")
        os.makedirs(run_dir, exist_ok=True)

        from datetime import datetime
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        print(f"[SETUP] size={size} | K={K} | run_dir={run_dir}")

        # Deterministic k-shots per (test_id,K) across models
        kshots_map = {}
        kshot_ids_map = {}
        for idx, test_row in tests_iter.iterrows():
            shots = select_kshots(test_row, K)
            kshots_map[test_ids[idx]] = shots
            kshot_ids_map[test_ids[idx]] = [int(getattr(s, "id", s.get("id", -1))) for s in shots]

        preds_by_mode = {0: {}, 1: {}}
        out_paths = {}
        tsv_paths = {}

        for hints_flag in WITH_HINTS:
            rows_out = []
            print(f"==> Running size={size} | K={K} | hints={hints_flag} | tests={len(tests_iter)}")

            out_path = os.path.join(
                run_dir,
                f"run_{stamp}_model-{MODEL_SLUG}_size-{size}_K-{K}_hints-{int(hints_flag)}.csv"
            )
            out_paths[int(hints_flag)] = out_path

            for idx, test_row in tests_iter.iterrows():
                test_id = test_ids[idx]
                kshots = kshots_map[test_id]
                user_prompt = build_user_prompt(test_row, kshots, with_hints=bool(hints_flag))
                try:
                    pred, raw, latency = call_lmstudio(user_prompt, SYSTEM_PROMPT, MODEL_ID)
                    err = 0
                except Exception as e:
                    pred, raw, latency, err = "", str(e), -1, 1

                preds_by_mode[int(hints_flag)][test_id] = pred

                rows_out.append({
                    "id": int(test_row.get("id", -1)),
                    "Source": nfc(test_row["source_quz"]),
                    "Change": nfc(test_row["change"]),
                    "gold_quz": nfc(test_row["target_quz"]),
                    "pred_quz": pred,
                    "with_hints": int(hints_flag),
                    "model_id": MODEL_ID,
                    "K": K,
                    "split": "ALL",
                    "size": size,
                    "primary_change": test_row["primary"],
                    "kshot_ids": json.dumps(kshot_ids_map[test_id], ensure_ascii=False),
                    "prompt_chars": len(user_prompt),
                    "latency_ms": latency,
                    "error_flag": err,
                })

            import pandas as pd
            df_out = pd.DataFrame(rows_out, columns=[
                "id","Source","Change","pred_quz","gold_quz",
                "with_hints","model_id","K","split","size","primary_change",
                "kshot_ids","prompt_chars","latency_ms","error_flag"
            ])
            df_out.to_csv(out_path, index=False, encoding="utf-8")
            print(f"[OK] Guardado: {out_path} ({len(df_out)} filas)")
            run_rows.extend(rows_out)

            tsv_path = os.path.join(
                run_dir,
                f"st2_{stamp}_model-{MODEL_SLUG}_size-{size}_hints-{int(hints_flag)}.tsv"
            )
            tsv_df = pd.DataFrame({
                "Source": df_out["Source"],
                "Change": df_out["Change"],
                "Predicted Target": df_out["pred_quz"],
                "Target": df_out["gold_quz"],
            })
            tsv_df.to_csv(tsv_path, sep="\t", index=False, encoding="utf-8")
            tsv_paths[int(hints_flag)] = tsv_path
            print(f"[OK] ST2 TSV: {tsv_path} ({len(tsv_df)} filas)")

        trace_rows = []
        for idx, test_row in tests_iter.iterrows():
            test_id = test_ids[idx]
            shots = kshots_map[test_id]
            for j, s in enumerate(shots, start=1):
                chg_s = "{" + ", ".join([f"{k}:{v}" for k, v in getattr(s, "change_dict", s["change_dict"]).items()]) + "}"
                trace_rows.append({
                    "test_id": test_id,
                    "row_type": "kshot",
                    "shot_idx": j,
                    "source": nfc(getattr(s, "source_quz", s["source_quz"])),
                    "change": nfc(chg_s),
                    "target_ref": nfc(getattr(s, "target_quz", s["target_quz"])),
                    "pred_nohint": "",
                    "pred_hint": ""
                })
            chg_final = nfc(test_row["change"])
            pred0 = preds_by_mode.get(0, {}).get(test_id, "")
            pred1 = preds_by_mode.get(1, {}).get(test_id, "")
            trace_rows.append({
                "test_id": test_id,
                "row_type": "final",
                "shot_idx": "",
                "source": nfc(test_row["source_quz"]),
                "change": chg_final,
                "target_ref": nfc(test_row["target_quz"]),
                "pred_nohint": pred0,
                "pred_hint": pred1
            })

        trace_df = pd.DataFrame(trace_rows, columns=[
            "test_id","row_type","shot_idx","source","change","target_ref","pred_nohint","pred_hint"
        ])
        trace_path = os.path.join(run_dir, f"trace_{stamp}_model-{MODEL_SLUG}_size-{size}.csv")
        trace_df.to_csv(trace_path, index=False, encoding="utf-8")
        print(f"[OK] Traza: {trace_path} ({len(trace_df)} filas)")

if run_rows:
    from datetime import datetime
    all_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    all_path = os.path.join(RUNS_DIR, f"runs_all_{all_stamp}.csv")
    import pandas as pd
    pd.DataFrame(run_rows).to_csv(all_path, index=False, encoding="utf-8")
    print(f"[OK] Acumulado: {all_path} ({len(run_rows)} filas)")

[INFO] size=1 | tests seleccionados=80 (filtro primary=ALL, max/primary=10, max/size=80)
[SETUP] size=1 | K=5 | run_dir=D:/Documentos/ExpLLMs/runs\mistral-small-3.2-24b-instruct-2506\K5
==> Running size=1 | K=5 | hints=False | tests=80
[OK] Guardado: D:/Documentos/ExpLLMs/runs\mistral-small-3.2-24b-instruct-2506\K5\run_20251101_005914_model-mistral-small-3.2-24b-instruct-2506_size-1_K-5_hints-0.csv (80 filas)
[OK] ST2 TSV: D:/Documentos/ExpLLMs/runs\mistral-small-3.2-24b-instruct-2506\K5\st2_20251101_005914_model-mistral-small-3.2-24b-instruct-2506_size-1_hints-0.tsv (80 filas)
==> Running size=1 | K=5 | hints=True | tests=80
[OK] Guardado: D:/Documentos/ExpLLMs/runs\mistral-small-3.2-24b-instruct-2506\K5\run_20251101_005914_model-mistral-small-3.2-24b-instruct-2506_size-1_K-5_hints-1.csv (80 filas)
[OK] ST2 TSV: D:/Documentos/ExpLLMs/runs\mistral-small-3.2-24b-instruct-2506\K5\st2_20251101_005914_model-mistral-small-3.2-24b-instruct-2506_size-1_hints-1.tsv (80 filas)
[OK] Traza: D:/Do